In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("updated_at", StringType(), True)
])

day1_data = [
    ("ORD-101", "CUST-A", 120.00, "Pending", "2026-11-23"),
    ("ORD-102", "CUST-B", 250.50, "Shipped", "2026-11-23"),
    ("ORD-103", "CUST-C",  85.00, "Pending", "2026-11-23")
]

day1_df = spark.createDataFrame(day1_data, schema)

# Save as a managed Delta table in Unity Catalog
day1_df.write.format("delta").mode("overwrite").saveAsTable("dbw_medallion_prod.default.silver_orders_practice")

display(spark.table("dbw_medallion_prod.default.silver_orders_practice"))

In [0]:
day2_data = [
    ("ORD-101", "CUST-A", 120.00, "Delivered", "2026-11-24"),  # Updated row
    ("ORD-104", "CUST-D", 310.00, "Pending",   "2026-11-24")   # New row
]

day2_df = spark.createDataFrame(day2_data, schema)
display(day2_df)

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, "dbw_medallion_prod.default.silver_orders_practice")

(
    target_table.alias("target")
    .merge(
        day2_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

display(spark.table("dbw_medallion_prod.default.silver_orders_practice").orderBy("order_id"))

In [0]:
%sql DESCRIBE HISTORY dbw_medallion_prod.default.silver_orders_practice

Cells 1–4 (SCD Type 1): Overwrites the row in place, so only the latest value (Delivered) stays in the table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

dim_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("effective_start_date", StringType(), True),
    StructField("effective_end_date", StringType(), True),
    StructField("is_current", BooleanType(), True)
])

day1_customers = [
    ("CUST-A", "Calgary",   "2026-01-01", "9999-12-31", True),
    ("CUST-B", "Vancouver", "2026-01-01", "9999-12-31", True)
]

spark.createDataFrame(day1_customers, dim_schema) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("dbw_medallion_prod.default.dim_customers_scd2")

display(spark.table("dbw_medallion_prod.default.dim_customers_scd2"))

In [0]:
updates_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("updated_at", StringType(), True)
])

day2_updates = [
    ("CUST-A", "Toronto",   "2026-11-24"),  # Changed city
    ("CUST-B", "Vancouver", "2026-11-24"),  # Unchanged
    ("CUST-C", "Montreal",  "2026-11-24")   # Brand-new customer
]

updates_df = spark.createDataFrame(day2_updates, updates_schema)
display(updates_df)

In [0]:
from delta.tables import DeltaTable

target_dim = DeltaTable.forName(spark, "dbw_medallion_prod.default.dim_customers_scd2")
active_target_df = target_dim.toDF().filter("is_current = true")

# 1. Find existing active customers whose city actually changed
changed_rows_df = (
    updates_df.alias("u")
    .join(active_target_df.alias("t"), on="customer_id", how="inner")
    .filter("u.city <> t.city")
    .select("u.*")
)

# 2. Stage updates:
#    - Top half (merge_key = customer_id): expires old rows & inserts brand-new customers
#    - Bottom half (merge_key = NULL): forces INSERT of the new version for changed customers
staged_updates_df = (
    updates_df.select(F.col("customer_id").alias("merge_key"), "*")
    .unionByName(
        changed_rows_df.select(F.lit(None).cast("string").alias("merge_key"), "*")
    )
)

# 3. Execute atomic SCD Type 2 MERGE
(
    target_dim.alias("target")
    .merge(
        staged_updates_df.alias("source"),
        "target.customer_id = source.merge_key"
    )
    .whenMatchedUpdate(
        condition="target.is_current = true AND target.city <> source.city",
        set={
            "is_current": "false",
            "effective_end_date": "source.updated_at"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "city": "source.city",
            "effective_start_date": "source.updated_at",
            "effective_end_date": "'9999-12-31'",
            "is_current": "true"
        }
    )
    .execute()
)

display(
    spark.table("dbw_medallion_prod.default.dim_customers_scd2")
    .orderBy("customer_id", "effective_start_date")
)

Cells 5–7 (SCD Type 2): Keeps both the old and new data—the old row (Calgary, is_current = false) stays preserved for history, and the new row (Toronto, is_current = true) is inserted right alongside it.